# Stage 1: DNA, RNA, and Protein Encoders on GTEx Transcript Expression

This notebook repeats the Stage 1 representation analysis on the transcript-expression dataset released with [IsoFormer](../Papers/Multi-Modal-Transfer-Learning.pdf). The [dataset](https://huggingface.co/datasets/InstaDeepAI/multi_omics_transcript_expression) is built from GTEx v8 transcript quantifications of protein-coding genes and contains 91,241 transcripts from 19,609 genes, split by gene into training, test, and validation sets. Each transcript's label is its expression in 30 tissues, averaged across donors and log-transformed as $\log(x + 1)$.

In the mRNA stability data used by [Stage1_refactor.ipynb](Stage1_refactor.ipynb) and [Stage1_multimodel.ipynb](Stage1_multimodel.ipynb), every model input derives from one coding sequence. Here, each transcript supplies inputs that carry different information:

| Input | Content | Information the other inputs lack |
|---|---|---|
| DNA | Genomic sequence from the hg38 reference, centered on the transcription start site | Promoter and other regulatory sequence outside the transcript |
| RNA | The full transcript, with the 5′ UTR, coding sequence, and 3′ UTR also provided separately | Untranslated-region sequence, which the protein does not reflect |
| Protein | The translated protein | None beyond the coding sequence; it is the coding sequence's translation |

The encoders match Stage1_multimodel.ipynb. The cells below outline the planned analyses. Code is added section by section.

## Research questions

- **Track A: Decodability.** How much of tissue expression can a linear probe recover from each encoder's frozen embedding, and from concatenated embeddings?
- **Track B: Cross-modal geometry.** For every pair of encoders, how similarly do their embedding spaces organize the same transcripts, before and after removing sequence composition?
- **Track C: Selected diagnostics.** With untranslated regions present, does attention in the RNA encoders concentrate at candidate stability motifs?

## Role in the project

The stability data define modalities by encoder, because all inputs come from one coding sequence. This dataset provides modalities that carry distinct information. Comparing the two tests whether frozen-encoder agreement, and the share of it that composition reproduces, differs between derived and distinct modalities, and whether those measurements anticipate when fusion adds information. On this task, IsoFormer reports $R^2 = 0.43$ with all three modalities and $0.36$ with RNA alone (Table 2).

---

## Setup

### Imports

Setup imports the Stage 1 analysis functions shared with the other Stage 1 notebooks, so every notebook computes each metric with the same implementation.

### Analysis settings

The settings define the transcript sample, the DNA window length for each DNA encoder, and the encoder list shared with Stage1_multimodel.ipynb.

---

## Data

### Downloading the transcript table

The dataset's Hugging Face loading script extracts a 200 kb DNA window from hg38 for every transcript. This notebook downloads the transcript table and extracts DNA only for the sampled transcripts.

### Sampling transcripts

The sample keeps one transcript per gene, so that closely related transcripts of the same gene do not inflate agreement or probe scores.

### DNA windows around the transcription start site

Each DNA window is centered on the transcription start site and cropped to the input limit of each DNA encoder.

### Inputs for each encoder

This section records which field each encoder reads (DNA window, full transcript, coding sequence, or protein) and how inputs longer than an encoder's token limit are truncated.

---

## Embeddings

### Embedding the sample with every encoder

Each sampled transcript is embedded by every encoder, using the mean-pooled final hidden state. Each encoder's embedding matrix is saved separately.

---

## Representation analysis

### Track A: Decodability

#### Expression probes across tissues

A multi-output linear probe predicts expression in all 30 tissues from each encoder's embedding. Results are summarized as $R^2$ averaged across tissues.

#### Sequence-property probes

Probes for length and GC content of the DNA window, untranslated regions, and coding sequence check what each encoder represents from its input.

#### Concatenated embeddings

Probes on concatenated embeddings test whether combining encoders adds linearly accessible expression information beyond the best single encoder.

---

### Track B: Cross-modal geometry

#### Pairwise similarity

Linear CKA, mutual $k$-nearest neighbors, CCA with cross-modal retrieval, and RSA with a Mantel permutation test are computed for every pair of encoders and reported as encoder-by-encoder matrices.

#### Composition control

The composition features extend those of Stage1_refactor.ipynb: coding-sequence codon and amino-acid frequencies, untranslated-region length and GC content, and DNA-window composition. Each similarity is reported before and after removing them.

---

### Track C: Selected diagnostics

#### Attention at candidate stability motifs with untranslated regions

AU-rich elements and m$^6$A sites concentrate in 3′ UTRs, which the stability data do not include. This comparison repeats the attention-motif test on transcripts that contain them, within each RNA encoder's input limit.

---

## Comparison with the stability results

This section compares each encoder pair's agreement, and its change under the composition control, between the stability data and this dataset.

---

## Summary

The summary records what distinct modalities change about the Stage 1 findings and what they imply for defining alignment.